# Three-Lens IL System: Calibration from DAC Tables

This notebook calibrates the intermediate-lens (IL) system of a TEM from
pre-parsed DAC tables using an **analytic closed-form solver** for the two
imaging constraints ($A = m$, $B = 0$).

## Notebook structure

| Section | Purpose |
|---------|---------|
| **Data** | Load 16-bit-normalised DAC tables for imaging and camera-length modes |
| **Physics** | Derivation of the analytic solver |
| **CAD priors** | Distance estimates from pixel measurements of the microscope column |
| **Analytic functions** | `phi2_exact`, `phi1_exact`, `model_at_data`, `make_residuals` |
| **Fit** | Broad search over sign combinations and $\alpha$ seeds, followed by refinement |
| **Evaluation** | Model-vs-data plots |
| **Export** | Convert fitted distances to z-positions and write to `microscope.toml` |


## Physical setup

Three intermediate lenses (IL1, IL2, IL3) sit between the objective and the
projector. The microscope operates in two modes that share the same IL optics
but differ in object-plane distance $d_1$.

| Mode | Target quantity | Relation |
|------|----------------|----------|
| **Imaging** | system magnification $M_\text{sys}$ | $M_\text{sys} = M_\text{obj}\times M_\text{IL}\times M_\text{proj}$ |
| **Camera length** | $\text{CL}$ [mm] | $\text{CL} = f_\text{obj}\times M_\text{IL}\times M_\text{proj}$ |

Scale factors $s_\text{img}$, $s_\text{cam}$ absorb the unknown
$M_\text{obj}$, $M_\text{proj}$, and $f_\text{obj}$.


In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import os
import tomllib
import tomlkit
from pathlib import Path
import json
from scipy.optimize import least_squares, brentq

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_MEM_FRACTION"] = "0.1"

## DAC curves

In [ ]:
mags_ref = np.array([30000.0, 40000.0, 50000.0, 60000.0, 80000.0, 100000.0,
                     120000.0, 150000.0, 200000.0, 250000.0, 300000.0,
                     400000.0, 500000.0, 600000.0], dtype=float)

cl_ref = np.array([80.0, 100.0, 120.0, 150.0, 200.0, 250.0, 300.0, 400.0,
                   500.0, 600.0, 800.0, 1000.0, 1200.0, 1500.0, 2000.0], dtype=float)

x_img = np.array([
    [0.621612548828125, 0.637542724609375, 0.65020751953125, 0.656494140625,
     0.659820556640625, 0.65625, 0.65557861328125, 0.6538238525390625,
     0.6514434814453125, 0.6510009765625, 0.6485595703125,
     0.64654541015625, 0.6449737548828125, 0.643798828125],
    [0.379364013671875, 0.379364013671875, 0.3929443359375, 0.4151611328125,
     0.443389892578125, 0.489349365234375, 0.489349365234375,
     0.5163726806640625, 0.5839080810546875, 0.5839080810546875,
     0.620208740234375, 0.67596435546875, 0.72369384765625,
     0.7647552490234375],
    [0.6062774658203125, 0.5423583984375, 0.5223236083984375,
     0.4974517822265625, 0.468048095703125, 0.4453887939453125,
     0.4238128662109375, 0.398040771484375, 0.360504150390625,
     0.3326873779296875, 0.2955322265625, 0.242523193359375,
     0.1968994140625, 0.1540679931640625],
], dtype=float)

x_cam = np.array([
    [0.3143310546875, 0.3218536376953125, 0.3208160400390625,
     0.325164794921875, 0.325164794921875, 0.3262176513671875,
     0.3267974853515625, 0.3269500732421875, 0.3263397216796875,
     0.3261871337890625, 0.3243865966796875, 0.3243865966796875,
     0.3234100341796875, 0.3220367431640625, 0.3204345703125],
    [0.6826324462890625, 0.6861724853515625, 0.690277099609375,
     0.6940460205078125, 0.70257568359375, 0.7133636474609375,
     0.7209320068359375, 0.7436981201171875, 0.7699737548828125,
     0.785888671875, 0.822265625, 0.8544921875, 0.8896484375,
     0.93798828125, 0.999755859375],
    [0.57269287109375, 0.5672454833984375, 0.5620269775390625,
     0.55426025390625, 0.5417327880859375, 0.529632568359375,
     0.5180511474609375, 0.4975128173828125, 0.4763641357421875,
     0.45703125, 0.42138671875, 0.390625, 0.358642578125,
     0.311767578125, 0.216796875],
], dtype=float)

print(f"Imaging:  {len(mags_ref)} points,  mag {mags_ref.min():.0f}..{mags_ref.max():.0f}")
print(f"Camera:   {len(cl_ref)} points,  CL {cl_ref.min():.0f}..{cl_ref.max():.0f} mm")
print("Compact data loaded. Proceed directly to distance fit (cell 4).")

## Analytical Solver (Physics Explainer)

### Magnetic lens optics recap

The optical power (inverse focal length) of a magnetic round lens is

$$\phi \;=\; \frac{e}{8\,m_e\,U^*}\;\int_{-\infty}^{\infty} B_z^2(z)\,dz$$

where $U^* = U(1 + eU/2m_ec^2)$ is the relativistic accelerating potential.
Since $B_z \propto NI$, we get $\phi \propto (NI)^2$ with a proportionality
constant that depends only on geometry (bore, gap, pole pieces):

$$\phi_k \;=\; \underbrace{\alpha_{\mathrm{geom},k} N_k^2 I_{\max}^2}_{\alpha_k}\, x_k^2$$

The fit uses lumped constants $\alpha_k$. If lens geometries are similar,
$\alpha_k$ differences mainly reflect turns-ratio differences.

### Transfer-matrix model

Each thin lens and drift is

$$P(d)=\begin{pmatrix}1 & d\\0 & 1\end{pmatrix},\qquad
L(\phi)=\begin{pmatrix}1 & 0\\-\phi & 1\end{pmatrix}$$

and the IL block is

$$M=P(d_4)L(\phi_3)P(d_3)L(\phi_2)P(d_2)L(\phi_1)P(d_1).$$

Define $A=M_{00}$ and $B=M_{01}$.

For an imaging solution at target IL magnification $m$, we require:

- magnification constraint: $A=m$
- focus constraint: $B=0$

### Why the solver is analytic

Write

$$M = M_{\text{rest}}\,L(\phi_1)\,P(d_1),$$

with $M_{\text{rest}}=P(d_4)L(\phi_3)P(d_3)L(\phi_2)P(d_2)$.
Then top-row terms can be written as

$$A = a - b\phi_1,\qquad B = a d_1 + b(1-d_1\phi_1),$$

where $a,b$ depend only on $(\phi_2,\phi_3)$.
Substitute $\phi_1=(a-m)/b$ from $A=m$ into $B=0$:

$$a d_1 + b - d_1(a-m)=0 \Rightarrow b + d_1 m = 0.$$

So $\phi_1$ cancels out, and we first solve

$$M_{\text{rest}}[0,1] = -d_1 m.$$

This is linear in $\phi_2$ for fixed $\phi_3$, giving a closed form used in code
(`phi2_exact`). Then $\phi_1$ follows from $A=m$ in another closed form
(`phi1_exact`).

### Inverse fit strategy used here

At each measured operating point, we use the measured $x_3$ to set
$\phi_3=\alpha_3 x_3^2$, then compute:

1. $\phi_2^*=\texttt{phi2\_exact}(\phi_3,m,\mathbf d)$
2. $\phi_1^*=\texttt{phi1\_exact}(\phi_2^*,\phi_3,m,\mathbf d)$
3. $x_1^*=\sqrt{\phi_1^*/\alpha_1}$ and $x_2^*=\sqrt{\phi_2^*/\alpha_2}$

Residuals compare $(x_1^*,x_2^*)$ against measured $(x_1,x_2)$.
By construction, each valid point satisfies $A=m$ and $B=0$ exactly.

### Parameters fitted

Nine fit parameters are used:

- distances: $d_1^{img}, d_1^{cam}, d_2, d_4$
- lens constants: $\alpha_1,\alpha_2,\alpha_3$
- mode scales: $s_{img}, s_{cam}$

### Workflow intent

This notebook now runs in a distance-first way:

1. estimate distances and lens constants
2. report distance analytics first
3. evaluate model-vs-data with plots afterward

## Read microscope geometry

In [ ]:
with open("microscope.toml", "rb") as fp:
    microscope_data = tomllib.load(fp)
microscope_data["modes"]["imaging"] = {}
print(json.dumps(microscope_data, indent=2))

In [ ]:
# Upstream z-positions (metres)
Z_SOURCE       = microscope_data["beam"]["z_source_m"]
Z_CL1          = microscope_data["lenses"]["CL1"]["z_m"]
Z_CL3          = microscope_data["lenses"]["CL3"]["z_m"]
Z_APERTURE     = microscope_data["apertures"]["C_aperture"]["z_m"]
Z_CMINI        = microscope_data["lenses"]["C_mini"]["z_m"]
Z_OBJ_PREFIELD = microscope_data["lenses"]["Obj_prefield"]["z_m"]
Z_SAMPLE       = microscope_data["sample"]["z_m"]
Z_OBJ_POST     = microscope_data["lenses"]["Obj_post"]["z_m"]

In [ ]:
OBJ_POST_F_MM = 2.273061  # Obj_post focal length (mm)
U_OBJ_MM = (Z_OBJ_POST - Z_SAMPLE) * 1000
OBJ_POST_V_MM = (OBJ_POST_F_MM * U_OBJ_MM) / (U_OBJ_MM - OBJ_POST_F_MM)
print(f"{OBJ_POST_V_MM:.2f} mm from Obj_post to obj_post image plane")

In [ ]:
OBJ_POST_TURNS = microscope_data["lenses"]["Obj_post"]["turns"]  # number of coil turns
OBJ_POST_Gc = microscope_data["lenses"]["Obj_post"]["Gc"]  # coil constant

In [ ]:
PL1_I     = -3.116334138300715    # coil current [A] (constant across modes)
PL1_TURNS = microscope_data["lenses"]["PL1"]["turns"]  # number of coil turns

In [ ]:
Z_SAMPLE_MM = Z_SAMPLE * 1000
Z_OBJ_POST_MM = Z_OBJ_POST * 1000
Z_IL1_MM = microscope_data["lenses"]["IL1"]["z_m"] * 1000
Z_IL2_MM = microscope_data["lenses"]["IL2"]["z_m"] * 1000
Z_IL3_MM = microscope_data["lenses"]["IL3"]["z_m"] * 1000
Z_PL1_MM = microscope_data["lenses"]["PL1"]["z_m"] * 1000
Z_SCREEN_MM = microscope_data["detector"]["z_m"] * 1000
D3_MM = (Z_IL3_MM - Z_IL2_MM)

CAD = {
    'd2':     (Z_IL2_MM - Z_IL1_MM) / D3_MM,   # IL1→IL2
    'd4':     (Z_PL1_MM - Z_IL3_MM) / D3_MM,   # IL3→projector/PL1
    'd1_img': (Z_IL1_MM - (Z_OBJ_POST_MM + OBJ_POST_V_MM))  / D3_MM,   # obj_post_image_plane→IL1 (imaging mode)
    'd1_cam': (Z_IL1_MM - (Z_OBJ_POST_MM + OBJ_POST_F_MM))  / D3_MM,   # obj_post_bfp→IL1 (camera-length mode)
}
CAD_SAMPLE_TO_DET_MM = (Z_SCREEN_MM - Z_SAMPLE_MM)
CAD_PL1_TO_DET_MM = (Z_SCREEN_MM - Z_PL1_MM)
U_OBJ_M = U_OBJ_MM * 1e-3  # sample -> Obj_post distance used for objective geometry tie

In [ ]:
print(f"Reference gap  d3 = {D3_MM:.1f} mm")
print(f"\nCAD distance priors (normalised to d3 = 1):")
for k, v in CAD.items():
    print(f"  {k:7s} = {v:.3f}")
print(f"\nShared CAD distances:")
print(f"  sample->detector = {CAD_SAMPLE_TO_DET_MM:.1f} mm")
print(f"  PL1->detector    = {CAD_PL1_TO_DET_MM:.1f} mm")
# print("""
# ===================
# Reference gap  d3 = 62.5 mm

# CAD distance priors (normalised to d3 = 1):
#   d2      = 1.233
#   d4      = 0.833
#   d1_img  = 0.200
#   d1_cam  = 3.200

# Shared CAD distances:
#   sample->detector = 718.8 mm
#   PL1->detector    = 325.0 mm
# """)

In [ ]:
# Prior weights: how strongly to pull fitted distances toward CAD values.
# d1_img prior is zeroed because the CAD estimate rests on only 6 pixels.
CAD_SIGMA = 0.20       # fractional uncertainty on each CAD distance
W_CAD     = 0.35       # weight for d2, d4 priors
W_CAD_D1I = 0.0        # weight for d1_img prior (disabled)

## Analytic solver functions

These two functions implement the closed-form solutions derived in the physics
cell above. For given lens power $\phi_3$, target IL magnification $m$, and
distances $\mathbf{d} = (d_1, d_2, d_3, d_4)$:

- **`phi2_exact`** solves the combined $B = 0$, $A = m$ constraint for $\phi_2$
- **`phi1_exact`** solves $A = m$ for $\phi_1$ once $\phi_2$ and $\phi_3$ are known

In [ ]:
# ── Closed-form solutions for phi2 and phi1 ──────────────────────────

def phi2_exact(phi3, m, d):
    """Solve the combined B=0, A=m constraint for phi2.

    Given phi3, target magnification m, and distances d = (d1, d2, d3, d4),
    returns the unique phi2 satisfying both imaging constraints.
    """
    d1, d2, d3, d4 = d
    num = d1 * m + (d2 + d3 + d4) - (d2 + d3) * d4 * phi3
    den = d2 * ((d3 + d4) - d3 * d4 * phi3)
    return num / den if abs(den) > 1e-30 else np.nan


def phi1_exact(phi2, phi3, m, d):
    """Solve A=m for phi1 with phi2, phi3 already known.

    Returns the lens power phi1 that gives the target magnification m.
    """
    d1, d2, d3, d4 = d
    A0 = 1.0 - (d3 + d4) * phi2 - d4 * phi3 + d3 * d4 * phi2 * phi3
    A1 = -(d2 + d3 + d4) + d2 * (d3 + d4) * phi2 + (d2 + d3) * d4 * phi3 - d2 * d3 * d4 * phi2 * phi3
    return (m - A0) / A1 if abs(A1) > 1e-30 else np.nan


def model_at_data(m_arr, x3_data, d, alpha):
    """Evaluate model-predicted DAC values at calibration points.

    For each operating point, fixes x3 to the measured value, then uses
    phi2_exact and phi1_exact to compute the predicted x1, x2.

    Returns (3, N) array of predicted [x1, x2, x3] DAC values.
    """
    out = np.full((3, len(m_arr)), np.nan)
    for j, m in enumerate(m_arr):
        p3 = alpha[2] * x3_data[j]**2
        p2 = phi2_exact(p3, m, d)
        if (not np.isfinite(p2)) or p2 <= 0:
            continue
        p1 = phi1_exact(p2, p3, m, d)
        if (not np.isfinite(p1)) or p1 <= 0:
            continue
        out[0, j] = np.sqrt(p1 / alpha[0])
        out[1, j] = np.sqrt(p2 / alpha[1])
        out[2, j] = x3_data[j]
    return out

In [ ]:
BALANCE_EXPONENT = 0.35

def _solve_m_obj_from_delta(delta, u):
    """Infer |M_obj| from (d1_cam - d1_img) and sample-to-objective distance."""
    if delta <= 0.0 or u <= 0.0:
        return None
    r = delta / u
    disc = r * r + 4.0 * r
    if disc <= 0.0:
        return None
    return 0.5 * (r + np.sqrt(disc))


def _derive_s_cam_from_s_img(d1i, d1c, s_img, d3_mm, u_obj_m=U_OBJ_M):
    """Derive s_cam from s_img using shared objective/projector geometry.

    This enforces one fixed projector magnification for both imaging and
    diffraction rather than fitting s_img and s_cam independently.
    """
    delta_m = (d1c - d1i) * d3_mm * 1e-3
    m_obj_abs = _solve_m_obj_from_delta(delta_m, u_obj_m)
    if m_obj_abs is None:
        return None

    v_obj_m = m_obj_abs * u_obj_m
    f_obj_m = (u_obj_m * v_obj_m) / (u_obj_m + v_obj_m)
    f_obj_mm = f_obj_m * 1e3
    return s_img * (m_obj_abs / f_obj_mm)


def plateau_weights(values):
    """Downweight repeated DAC plateaus caused by table quantisation."""
    unique_vals, counts = np.unique(values, return_counts=True)
    count_map = dict(zip(unique_vals.tolist(), counts.tolist()))
    return np.array([count_map[val]**-0.5 for val in values], dtype=float)


# Balance the fit so flatter DAC curves are not overwhelmed by the higher-span ones.
img_span = np.ptp(x_img, axis=1)
cam_span = np.ptp(x_cam, axis=1)
img_weights = (np.median(img_span) / np.maximum(img_span, 1e-6))**BALANCE_EXPONENT
cam_weights = (np.median(cam_span) / np.maximum(cam_span, 1e-6))**BALANCE_EXPONENT

# Additional pointwise weights reduce the leverage of repeated DAC plateaus.
img_plateau_weights = np.vstack([plateau_weights(x_img[k]) for k in range(3)])
cam_plateau_weights = np.vstack([plateau_weights(x_cam[k]) for k in range(3)])
img_point_weights = img_weights[:, None] * img_plateau_weights
cam_point_weights = cam_weights[:, None] * cam_plateau_weights


def make_residuals(si, sc, w_cad_d1i=W_CAD_D1I):
    """Build the residual function for least_squares.

    Parameters
    ----------
    si, sc : +1 or -1
        Sign of the IL magnification in imaging / camera-length mode.
    w_cad_d1i : float
        Weight for the d1_img CAD prior (can be adjusted between stages).

    The 8-element parameter vector is:
        [d1_img, d1_cam, d2, d4, log(alpha1), log(alpha2), log(alpha3),
         log(s_img)]

    s_cam is derived from s_img and the objective geometry implied by
    (d1_cam - d1_img), enforcing one fixed projector scale across modes.
    """
    def fn(p):
        d1i, d1c, d2, d4 = p[:4]
        alpha = np.exp(p[4:7])
        s_i = np.exp(p[7])
        s_c = _derive_s_cam_from_s_img(d1i, d1c, s_i, D3_MM)
        if s_c is None or (not np.isfinite(s_c)) or s_c <= 0.0:
            return np.full(2 * (x_img.shape[1] + x_cam.shape[1]) + 3, 1e3)

        d_i, d_c = [d1i, d2, 1.0, d4], [d1c, d2, 1.0, d4]
        res = []
        for x_data, targets, d, s, sign, weights in [
            (x_img, mags_ref, d_i, s_i, si, img_point_weights),
            (x_cam, cl_ref,   d_c, s_c, sc, cam_point_weights),
        ]:
            for j in range(x_data.shape[1]):
                m = sign * s * targets[j]
                p3 = alpha[2] * x_data[2, j]**2
                p2s = phi2_exact(p3, m, d)
                if (not np.isfinite(p2s)) or p2s <= 0:
                    res.extend([1.0, 1.0])
                    continue
                p1s = phi1_exact(p2s, p3, m, d)
                if (not np.isfinite(p1s)) or p1s <= 0:
                    res.extend([1.0, 1.0])
                    continue
                res.append(weights[0, j] * (np.sqrt(p1s / alpha[0]) - x_data[0, j]))
                res.append(weights[1, j] * (np.sqrt(p2s / alpha[1]) - x_data[1, j]))
        # CAD distance priors
        for val, ref in [(d2, CAD['d2']), (d4, CAD['d4'])]:
            res.append(W_CAD * (val - ref) / (CAD_SIGMA * ref))
        res.append(w_cad_d1i * (d1i - CAD['d1_img']) / (CAD_SIGMA * CAD['d1_img']))
        res.append(W_CAD / 3 * (d1c - CAD['d1_cam']) / (CAD_SIGMA * CAD['d1_cam']))
        return np.array(res)
    return fn


# Parameter bounds
lb = np.array([0.05, 0.10, 0.50, 0.30,
               np.log(0.01), np.log(0.01), np.log(0.01),
               np.log(1e-7)])
ub = np.array([0.80, 7.0, 2.60, 1.80,
               np.log(1000), np.log(1000), np.log(1000),
               np.log(1.0)])

# Initial guess seeded from CAD distances
p0_base = [CAD['d1_img'], CAD['d1_cam'], CAD['d2'], CAD['d4'],
           0, 0, 0, np.log(1/3000)]

print("Analytic functions and residual builder defined.")
print(f"Parameter bounds: {len(lb)} parameters")
print(f"  Distances: d1_img in [{lb[0]:.2f}, {ub[0]:.2f}], d1_cam in [{lb[1]:.2f}, {ub[1]:.2f}]")
print(f"  d2 in [{lb[2]:.2f}, {ub[2]:.2f}], d4 in [{lb[3]:.2f}, {ub[3]:.2f}]")
print(f"Residual weighting exponent = {BALANCE_EXPONENT:.2f}")
print(f"Residual weighting  imaging = {img_weights.round(3)},  camera = {cam_weights.round(3)}")
print(f"Plateau weighting   imaging repeats = {np.unique(img_plateau_weights).round(3)},  camera repeats = {np.unique(cam_plateau_weights).round(3)}")

## Fit: Broad search + refinement

Run a broad search over all four sign combinations and multiple $\alpha$ seeds,
then refine the best result. The `d1_img` CAD prior is **disabled** (`W_CAD_D1I = 0`)
so the fit can explore freely.


In [ ]:
# ── Fit: broad search + refinement (d1_img prior disabled) ───────────

best, best_cost, best_signs = None, np.inf, (+1, +1)

for si, sc in [(+1, +1), (+1, -1), (-1, +1), (-1, -1)]:
    fn = make_residuals(si, sc, w_cad_d1i=0.0)
    for a1, a2, a3 in [(10, 10, 5), (3, 3.5, 2.5), (20, 20, 10), (5, 7, 4)]:
        p0 = list(p0_base)
        p0[4:7] = [np.log(a1), np.log(a2), np.log(a3)]
        res = least_squares(fn, p0, bounds=(lb, ub), method='trf',
                            max_nfev=20000, diff_step=0.001)
        if res.cost < best_cost:
            best_cost, best, best_signs = res.cost, res, (si, sc)
        else:
            print(f"Sign combo (si={si:+d}, sc={sc:+d}) with a={a1},{a2},{a3} → cost {res.cost:.6f} (not better than {best_cost:.6f})")

si_best, sc_best = best_signs

# Refine the best result with tight tolerances
fn_final = make_residuals(*best_signs, w_cad_d1i=0.0)
final = least_squares(fn_final, best.x, bounds=(lb, ub), method='trf',
                      max_nfev=100000, diff_step=1e-4,
                      ftol=1e-15, xtol=1e-15, gtol=1e-15)
if final.cost > best.cost:
    final = best

In [ ]:
# Extract final fitted parameters
d1i_f, d1c_f, d2_f, d4_f = final.x[:4]
alpha_f = np.exp(final.x[4:7])
s_img_f = np.exp(final.x[7])
s_cam_f = _derive_s_cam_from_s_img(d1i_f, d1c_f, s_img_f, D3_MM)
if s_cam_f is None or (not np.isfinite(s_cam_f)) or s_cam_f <= 0.0:
    raise ValueError("Derived s_cam is invalid for final fit parameters")

d_img_f = np.array([d1i_f, d2_f, 1.0, d4_f])
d_cam_f = np.array([d1c_f, d2_f, 1.0, d4_f])
m_img = si_best * s_img_f * mags_ref
m_cam = sc_best * s_cam_f * cl_ref

print(f"Fit result  signs=({si_best:+d},{sc_best:+d})  cost={final.cost:.6f}")
print(f"\nFitted distances (d3 = 1 = {D3_MM:.1f} mm):")
for nm, fit, cad in [('d1_img', d1i_f, CAD['d1_img']),
                     ('d1_cam', d1c_f, CAD['d1_cam']),
                     ('d2',     d2_f,  CAD['d2']),
                     ('d4',     d4_f,  CAD['d4'])]:
    print(f"  {nm:7s} = {fit:.4f}  ({fit * D3_MM:.2f} mm)  "
          f"CAD {cad:.3f} ({cad * D3_MM:.2f} mm)  "
          f"Δ = {(fit - cad) / (CAD_SIGMA * cad):+.1f}σ")

print(f"\nLens constants   α = [{alpha_f[0]:.4f}, {alpha_f[1]:.4f}, {alpha_f[2]:.4f}]")
print(f"Scale factors    s_img = {s_img_f:.4e}  →  M_obj·M_proj = {1/s_img_f:.1f}")
print(f"                 s_cam = {s_cam_f:.4e}  →  f_obj·M_proj = {1/s_cam_f:.1f}  (derived)")

## Model Evaluation

This section does not re-fit anything. It only evaluates the fitted model at the
measured operating points so we can inspect how the calibrated geometry and lens
constants reproduce the DAC tables.

To keep the presentation tighter while still being honest, each lens now gets:

- a full-scale panel on the native DAC range `[0, 1]`
- an inset zoom that shows the local mismatch clearly
- a residual panel below with `model - data`

This makes it easy to judge both the absolute operating range and the remaining
structured error without expanding the figure into separate full-scale and zoomed
grids.

In [ ]:
# ── Evaluate model at all calibration points ─────────────────────────

pi_f = model_at_data(m_img, x_img[2], d_img_f, alpha_f)
pc_f = model_at_data(m_cam, x_cam[2], d_cam_f, alpha_f)

n_img_ok = int(np.sum(np.isfinite(pi_f[0])))
n_cam_ok = int(np.sum(np.isfinite(pc_f[0])))

# Per-lens RMSE
for mode, pred, meas, n_ok, label in [
    ('Imaging', pi_f, x_img, n_img_ok, mags_ref),
    ('Camera',  pc_f, x_cam, n_cam_ok, cl_ref),
]:
    print(f"\n{mode} mode: {n_ok}/{pred.shape[1]} points valid")
    for k, lens in enumerate(['IL1', 'IL2', 'IL3']):
        ok = np.isfinite(pred[k])
        if np.any(ok):
            rmse = np.sqrt(np.mean((pred[k, ok] - meas[k, ok])**2))
            print(f"  {lens} RMSE = {rmse:.5f}")

In [ ]:
# ── Plot: compact model-vs-data dashboard (no zoom insets) ────────────

lens_names = ['IL1', 'IL2', 'IL3']
lens_colors = ['C0', 'C1', 'C2']

fig, axes = plt.subplots(4, 3, figsize=(15, 16))

log_mags = np.log10(mags_ref)
log_cl = np.log10(cl_ref)
img_xlim = (log_mags.min() - 0.05, log_mags.max() + 0.05)
cam_xlim = (log_cl.min() - 0.05, log_cl.max() + 0.05)


def residual_limits(residual, pad=0.15):
    values = residual[np.isfinite(residual)]
    if values.size == 0:
        return -0.05, 0.05
    bound = np.max(np.abs(values))
    bound = max(bound * (1 + pad), 0.01)
    return -bound, bound


def residual_stats(pred, data):
    ok = np.isfinite(pred)
    residual = np.full_like(pred, np.nan)
    residual[ok] = pred[ok] - data[ok]
    rmse = np.sqrt(np.mean(residual[ok]**2)) if np.any(ok) else np.nan
    bias = np.mean(residual[ok]) if np.any(ok) else np.nan
    return residual, rmse, bias


for k, (name, color) in enumerate(zip(lens_names, lens_colors)):
    residual_img, rmse_img, bias_img = residual_stats(pi_f[k], x_img[k])
    residual_cam, rmse_cam, bias_cam = residual_stats(pc_f[k], x_cam[k])

    ax = axes[0, k]
    ok = np.isfinite(pi_f[k])
    ax.plot(log_mags, x_img[k], 'o', color=color, ms=5, label='Data')
    ax.plot(log_mags[ok], pi_f[k, ok], 's--', color=color, ms=4, alpha=0.75, label='Model')
    ax.set_xlim(*img_xlim)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel('DAC / $2^{16}$')
    ax.set_title(f'{name} — Imaging')
    ax.grid(True, alpha=0.3)
    ax.text(0.98, 0.04, f'RMSE {rmse_img:.4f}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.8))
    if k == 0:
        ax.legend(fontsize=8, loc='upper left')

    ax = axes[1, k]
    ok = np.isfinite(pi_f[k])
    ax.axhline(0.0, color='k', lw=1.0, alpha=0.6)
    ax.plot(log_mags[ok], residual_img[ok], 'o--', color=color, ms=4)
    ylo, yhi = residual_limits(residual_img)
    ax.set_xlim(*img_xlim)
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel('Model - data')
    ax.set_xlabel('log$_{10}$(Magnification)')
    ax.set_title(f'{name} residual  (bias {bias_img:+.4f})')
    ax.grid(True, alpha=0.3)

    ax = axes[2, k]
    ok = np.isfinite(pc_f[k])
    ax.plot(log_cl, x_cam[k], 'o', color=color, ms=5, label='Data')
    ax.plot(log_cl[ok], pc_f[k, ok], 's--', color=color, ms=4, alpha=0.75, label='Model')
    ax.set_xlim(*cam_xlim)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel('DAC / $2^{16}$')
    ax.set_title(f'{name} — Camera length')
    ax.grid(True, alpha=0.3)
    ax.text(0.98, 0.04, f'RMSE {rmse_cam:.4f}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.8))
    if k == 0:
        ax.legend(fontsize=8, loc='upper left')

    ax = axes[3, k]
    ok = np.isfinite(pc_f[k])
    ax.axhline(0.0, color='k', lw=1.0, alpha=0.6)
    ax.plot(log_cl[ok], residual_cam[ok], 'o--', color=color, ms=4)
    ylo, yhi = residual_limits(residual_cam)
    ax.set_xlim(*cam_xlim)
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel('Model - data')
    ax.set_xlabel('log$_{10}$(CL [mm])')
    ax.set_title(f'{name} residual  (bias {bias_cam:+.4f})')
    ax.grid(True, alpha=0.3)

fig.suptitle('Final fit: compact model-vs-data dashboard', fontsize=15, y=0.985)
fig.subplots_adjust(top=0.93, bottom=0.05, left=0.06, right=0.98, hspace=0.22, wspace=0.30)
plt.show()


## Export fitted geometry for microscope.toml

Convert the fitted normalised distances to absolute z-positions (metres) consistent
with the column geometry used in `microscope_design_curves.ipynb` and
`microscope.toml`.

The constraint study fits distances normalised to $d_3 = 1$ (the IL2→IL3 gap).
To recover physical distances we multiply by `D3_MM` (mm) then convert to metres.

Critical convention:
- $d_1^{img}$ is measured from the **objective image plane** to IL1, not from Obj_post.
- The objective image plane is computed from a chosen objective magnification
  ($M_{obj}$) and the sample-to-objective distance.

With objective lens at $z_{obj}$, sample at $z_s$, and $u = z_{obj} - z_s$:

$$v = |M_{obj}|\,u, \qquad z_{obj\_image} = z_{obj} + v$$

Then the fitted IL coordinates are

$$z_{IL1} = z_{obj\_image} + d_1^{img}, \quad
z_{IL2} = z_{IL1} + d_2, \quad
z_{IL3} = z_{IL2} + d_3, \quad
z_{PL1} = z_{IL3} + d_4.$$

For diffraction mode, $d_1^{cam}$ is interpreted as BFP-to-IL1 and is printed
alongside the value implied by the chosen objective geometry for consistency checks.

### IL Distances

In [ ]:
# Fitted physical distances (metres).
d1_img_m = d1i_f * D3_MM * 1e-3   # objective_image_plane -> IL1 (imaging mode)
d1_cam_m = d1c_f * D3_MM * 1e-3   # BFP -> IL1 (camera-length mode)
d2_m = d2_f * D3_MM * 1e-3        # IL1 -> IL2
d3_m = D3_MM * 1e-3               # IL2 -> IL3 (reference gap)
d4_m = d4_f * D3_MM * 1e-3        # IL3 -> IL image plane (NOT PL1!)

### Objective

In [ ]:
# Objective geometry convention:
# d1_cam - d1_img = z_obj_image - z_BFP = v_obj - f_obj.
# Solve for |M_obj| from this fitted difference when possible.
u_obj_m = Z_OBJ_POST - Z_SAMPLE
delta_img_to_bfp_m = d1_cam_m - d1_img_m

M_OBJ_FOR_GEOMETRY = None  # Set a value like -30.0 to force it; None = infer from fitted distances.

if M_OBJ_FOR_GEOMETRY is None:
    m_obj_abs = _solve_m_obj_from_delta(delta_img_to_bfp_m, u_obj_m)
    if m_obj_abs is None:
        raise ValueError(
            "Cannot infer objective magnification from fitted d1 values: "
            f"d1_cam - d1_img = {delta_img_to_bfp_m*1e3:.3f} mm is not positive."
        )
    m_obj_signed = -float(m_obj_abs)
    objective_mode = "auto_from_fitted_d1"
else:
    m_obj_signed = float(M_OBJ_FOR_GEOMETRY)
    m_obj_abs = abs(m_obj_signed)
    objective_mode = "forced"

v_obj_m = m_obj_abs * u_obj_m  # distance to obj_post image plane
f_obj_m = (u_obj_m * v_obj_m) / (u_obj_m + v_obj_m)  # distance to obj_post BFP
z_bfp = Z_OBJ_POST + f_obj_m

print(f"Objective geometry ({objective_mode}): M_obj = {m_obj_signed:+.6f}")
print(f"  u = {u_obj_m*1e3:.3f} mm,  v = {v_obj_m*1e3:.3f} mm,  f = {f_obj_m*1e3:.6f} mm")

### Obj_post current

Note the -1 here since by convention the IL lenses are positive, obj + PL are negative

In [ ]:
OBJ_POST_I = -1 * (1 / OBJ_POST_TURNS) * np.sqrt(1 / (f_obj_m * OBJ_POST_Gc))  # coil current for Obj_post
print(f"Obj_post drive current: {OBJ_POST_I:.1f} A")

### IL Positions

In [ ]:
# Derive z-positions from objective-image convention.
z_il1 = Z_OBJ_POST + v_obj_m + d1_img_m
z_il2 = z_il1 + d2_m
z_il3 = z_il2 + d3_m
z_il_image = z_il3 + d4_m          # where B=0 for the 3-lens IL block

print("\nDerived z-positions (metres):")
print(f"  IL1: {z_il1:.6f}")
print(f"  IL2: {z_il2:.6f}")
print(f"  IL3: {z_il3:.6f}")
print(f"  IL image: {z_il_image:.6f}")

### Detector

In [ ]:
# Detector z-position from shared CAD geometry constants.
z_detector = Z_SAMPLE + CAD_SAMPLE_TO_DET_MM * 1e-3
print(f"Detector z-position: {z_detector:.6f} m")

### PL1

In [ ]:
# ── PL1: calibrate Gc from the fit's scale factor ────────────────────
# The fit determined s_img such that m_IL = si * s_img * mag_ref.
# The total magnification factors as M_total = M_obj * m_IL * M_PL1,
# so |M_PL1| = 1 / (|M_obj| * s_img).
NI_PL1    = PL1_TURNS * abs(PL1_I)

M_PL1_required = 1.0 / (m_obj_abs * s_img_f)

# Self-consistent PL1 position from required magnification:
# D = z_det - z_il_image (total IL_image -> Detector)
# u = D / (1 + |M_PL1|),  v = D - u,  f = u*v/D
D_img_to_det = z_detector - z_il_image
u_pl1_m = D_img_to_det / (1.0 + M_PL1_required)
v_pl1_m = D_img_to_det - u_pl1_m
f_pl1_m = u_pl1_m * v_pl1_m / D_img_to_det

# Derive PL1 Gc from the required focal length: 1/f = Gc * NI^2
Gc_PL1_calibrated = 1.0 / (f_pl1_m * NI_PL1**2)

z_pl1 = z_il_image + u_pl1_m
print(f"\nPL1:")
print(f"  Required PL1 magnification: {M_PL1_required:.3f}")
print(f"  Required PL1 focal length: {f_pl1_m*1e3:.3f} mm")
print(f"  Calibrated PL1 Gc: {Gc_PL1_calibrated:.6e} m/A^2")
print(f"  PL1 z-position: {z_pl1:.6f} m")

### IL Currents

In [ ]:
# ── Convert DAC fractions (x) to physical coil currents (Amps) ──────
# The fit uses normalised optical power:  phi_norm = alpha_k * x_k^2
# Physical optical power:                 phi_phys = phi_norm / d3_m
# TOML convention at reference voltage:   phi_phys = Gc_toml * (N * I)^2
# Therefore:  I_k = x_k * sqrt(alpha_k / (d3_m * Gc_toml * N^2))
d3_m_val = D3_MM * 1e-3  # d3 in metres

il_lens_names = ['IL1', 'IL2', 'IL3']
il_turns = np.array([float(microscope_data['lenses'][n]['turns']) for n in il_lens_names])
il_gc = np.array([float(microscope_data['lenses'][n]['Gc']) for n in il_lens_names])

# I_max_k converts x_k (DAC fraction) to I_k (Amps):  I_k = x_k * I_max_k
I_max = np.sqrt(alpha_f / (d3_m_val * il_gc * il_turns**2))

print("DAC -> current conversion factors (I_max per lens):")
for k, name in enumerate(il_lens_names):
    print(f"  {name}: I_max = {I_max[k]:.6f} A  (N={il_turns[k]:.0f}, Gc={il_gc[k]:.2e})")

# Convert model-predicted DAC arrays to physical currents
pi_f_amps = pi_f * I_max[:, None]   # imaging mode
pc_f_amps = pc_f * I_max[:, None]   # diffraction mode

### Summary

In [ ]:
# Consistency check against camera-mode convention (BFP -> IL1).
d1_cam_from_geom_m = z_il1 - z_bfp
d1_cam_delta_m = d1_cam_from_geom_m - d1_cam_m

print(f"\nFitted z-positions:")
print(f"  z_IL1 = {z_il1:.9f} m")
print(f"  z_IL2 = {z_il2:.9f} m")
print(f"  z_IL3 = {z_il3:.9f} m")
print(f"  z_IL_img = {z_il_image:.9f} m  (IL image plane)")
print(f"  z_PL1 = {z_pl1:.9f} m  (= z_IL_img + u_PL1)")
print(f"  z_Det = {z_detector:.9f} m")
print(f"  PL1->Det = {v_pl1_m*1e3:.1f} mm")
print(f"\nBFP-to-IL1 consistency: fitted = {d1_cam_m*1e3:.3f} mm, "
      f"geometry = {d1_cam_from_geom_m*1e3:.3f} mm, delta = {d1_cam_delta_m*1e3:+.3f} mm")

## Export IL tables for microscope_design_curves

Save the calibrated DAC tables and lens constants to an NPZ file so that
`microscope_design_curves.ipynb` can load them directly instead of
independently re-solving the IL operating points.

In [ ]:
# Write fitted IL operating tables and fitted lens positions back to microscope.toml.
_cwd = Path.cwd().resolve()
_repo = next((p for p in [_cwd, *_cwd.parents] if (p / 'src').exists()), _cwd)
toml_path = _repo / 'examples/microscope_models/microscope_fit.toml'

In [ ]:
# Update z-positions (computed in the export cell above).
microscope_data['lenses']['IL1']['z_m'] = z_il1
microscope_data['lenses']['IL2']['z_m'] = z_il2
microscope_data['lenses']['IL3']['z_m'] = z_il3
microscope_data['lenses']['PL1']['z_m'] = z_pl1
microscope_data['detector']['z_m'] = z_detector

In [ ]:
# Update PL1 Gc from the calibrated value (derived from fit scale factor).
microscope_data['lenses']['PL1']['Gc'] = float(Gc_PL1_calibrated)

In [ ]:
imaging_current_headers = ["Obj_post", "IL1", "IL2", "IL3", "PL1"]

# Update magnification table with physical IL currents.
mag_tbl = {"headers": ["mag"] + imaging_current_headers, "values": []}  # microscope_data['modes']['imaging']['magnification']
mag_idx = {k: i for i, k in enumerate(mag_tbl['headers'])}

new_mag_rows = []
for j in range(len(mags_ref)):
    row = []
    row.append(float(mags_ref[j]))
    row.append(float(OBJ_POST_I))
    row.append(float(pi_f_amps[0, j]))
    row.append(float(pi_f_amps[1, j]))
    row.append(float(pi_f_amps[2, j]))
    row.append(float(PL1_I))
    new_mag_rows.append(row)
mag_tbl['values'] = new_mag_rows
microscope_data['modes']['imaging']['magnification'] = mag_tbl

# Update diffraction table with physical IL currents.
# Persist camera_length in metres for microscope.toml compatibility.
diff_tbl = {"headers": ["camera_length"] + imaging_current_headers, "values": []}
diff_idx = {k: i for i, k in enumerate(diff_tbl['headers'])}

cam_control_m = cl_ref * 1e-3

new_diff_rows = []
for j in range(len(cam_control_m)):
    row = []
    row.append(float(cam_control_m[j]))
    row.append(float(OBJ_POST_I))
    row.append(float(pc_f_amps[0, j]))
    row.append(float(pc_f_amps[1, j]))
    row.append(float(pc_f_amps[2, j]))
    row.append(float(PL1_I))
    new_diff_rows.append(row)
diff_tbl['values'] = new_diff_rows
microscope_data['modes']['imaging']['diffraction'] = diff_tbl

In [ ]:
toml_path.write_text(
    tomlkit.dumps(microscope_data), encoding='utf-8',
)

In [ ]:
# Verify the written file parses correctly.
with open(toml_path, 'rb') as f:
    _ = tomllib.load(f)

## Reload checks

In [ ]:
from temgym_core.microscope_model import MicroscopeModel
# Post-write semantic validation through the actual model loader.
model = MicroscopeModel.from_toml(
    str(toml_path),
    mode_names={
        'imaging.magnification': 'mag',
        'imaging.diffraction': 'diff',
    },
)

lens_z = {cfg.name: cfg.z_position for cfg in model.lenses}
required_lenses = ('IL1', 'IL2', 'IL3', 'PL1')
if not all(name in lens_z for name in required_lenses):
    raise KeyError(f"Missing one of required lenses in model: {required_lenses}")

if not (lens_z['IL1'] < lens_z['IL2'] < lens_z['IL3'] < lens_z['PL1']):
    raise ValueError(
        "Lens z-order invalid: expected IL1 < IL2 < IL3 < PL1, got "
        f"{lens_z['IL1']:.9f}, {lens_z['IL2']:.9f}, {lens_z['IL3']:.9f}, {lens_z['PL1']:.9f}"
    )

if 'detector' not in model.auxiliary or 'z' not in model.auxiliary['detector']:
    raise KeyError("Detector z not found in parsed microscope model")
det_z = float(model.auxiliary['detector']['z'])
if not (det_z > lens_z['PL1']):
    raise ValueError(
        f"Detector must be downstream of PL1: det={det_z:.9f}, PL1={lens_z['PL1']:.9f}"
    )

for mode_name in ('mag', 'diff'):
    mode = model.modes[mode_name]
    if not np.all(np.isfinite(mode.control_values)):
        raise ValueError(f"Non-finite control values in mode '{mode_name}'")
    if not np.all(np.diff(mode.control_values) > 0):
        raise ValueError(f"Control values must be strictly increasing in mode '{mode_name}'")
    mid_control = float(mode.control_values[len(mode.control_values) // 2])
    mid_currents = mode.interpolate_currents(mid_control)
    if not np.all(np.isfinite(mid_currents)):
        raise ValueError(f"Non-finite interpolated currents in mode '{mode_name}' at {mid_control}")

if not np.isfinite(pi_f_amps).all() or not np.isfinite(pc_f_amps).all():
    raise ValueError("Non-finite IL current values generated for table export")

print(f"\nUpdated microscope config: {toml_path}")
print(f"  magnification rows: {len(new_mag_rows)}")
print(f"  diffraction rows:   {len(new_diff_rows)}  (camera_length unit: m)")
print(f"  lens z: IL1={z_il1:.9f}, IL2={z_il2:.9f}, IL3={z_il3:.9f}, PL1={z_pl1:.9f}")
print(f"  PL1 Gc: {Gc_PL1_calibrated:.6e}  (calibrated from fit)")
print(f"  detector z: {z_detector:.9f}  (PL1->Det = {v_pl1_m*1e3:.1f} mm)")
print(f"  M_obj = {m_obj_signed:+.6f} ({objective_mode})")
print(f"  |M_PL1| = {M_PL1_required:.2f}")
print(f"  BFP-to-IL1 consistency: delta = {d1_cam_delta_m*1e3:+.3f} mm")
print(f"\nSample IL1 current at mag=100k: DAC x={pi_f[0, 5]:.4f} -> I={pi_f_amps[0, 5]:.4f} A")
print("Post-write validation passed: TOML parse, model reload, ordering and interpolation checks.")

In [ ]:
# # Optional diagnostic: replay audit for loaded TOML vs fitted IL constraints.
# from temgym_core.components import SigmoidAperture, Detector, Plane
# from temgym_core.run import solve_model_with_z
# from temgym_core.plotting import _rotation_matrix_5x5
# from temgym_core.ray import Ray

# model_check = MicroscopeModel.from_toml(
#     str(toml_path),
#     mode_names={
#         'illumination.parallel': 'spot',
#         'imaging.magnification': 'mag',
#         'imaging.diffraction': 'diff',
#     },
#     mode_defaults={'spot': {'Obj_prefield': 1.0}},
# )

# lens_names = [cfg.name for cfg in model_check.lenses]
# il_names = ['IL1', 'IL2', 'IL3']
# il_idx = [lens_names.index(name) for name in il_names]

# def _mode_il_currents(mode_name, controls):
#     mode = model_check.modes[mode_name]
#     stack = np.vstack([mode.interpolate_currents(float(c)) for c in np.asarray(controls, dtype=float)])
#     return stack[:, il_idx].T

# # 1) Table-replay fidelity at mode control points
# mag_mode = model_check.modes['mag']
# diff_mode = model_check.modes['diff']

# mag_ctrl = np.asarray(mag_mode.control_values, dtype=float)
# diff_ctrl = np.asarray(diff_mode.control_values, dtype=float)

# mag_loaded_il = _mode_il_currents('mag', mag_ctrl)
# diff_loaded_il = _mode_il_currents('diff', diff_ctrl)

# mag_grid_ok = (mag_ctrl.shape == mags_ref.shape) and np.allclose(mag_ctrl, mags_ref, rtol=0.0, atol=0.0)
# diff_grid_ok = (diff_ctrl.shape == cl_ref.shape) and np.allclose(diff_ctrl, cl_ref * 1e-3, rtol=0.0, atol=0.0)

# if mag_grid_ok:
#     mag_replay_err = float(np.max(np.abs(mag_loaded_il - pi_f_amps)))
# else:
#     mag_interp_fit = np.vstack([np.interp(mag_ctrl, mags_ref, pi_f_amps[k]) for k in range(3)])
#     mag_replay_err = float(np.max(np.abs(mag_loaded_il - mag_interp_fit)))

# if diff_grid_ok:
#     diff_replay_err = float(np.max(np.abs(diff_loaded_il - pc_f_amps)))
# else:
#     diff_interp_fit = np.vstack([np.interp(diff_ctrl, cl_ref * 1e-3, pc_f_amps[k]) for k in range(3)])
#     diff_replay_err = float(np.max(np.abs(diff_loaded_il - diff_interp_fit)))

# print('Loaded-mode replay of written IL tables:')
# print(f'  imaging grid exact match:     {mag_grid_ok}')
# print(f'  diffraction grid exact match: {diff_grid_ok}')
# print(f'  max |imaging loaded-fit| [A]:     {mag_replay_err:.3e}')
# print(f'  max |diffraction loaded-fit| [A]: {diff_replay_err:.3e}')

# # 2) Coupling audit: does spot mode override diffraction IL drives?
# spot_mode = model_check.modes['spot']
# spot_curr = spot_mode.interpolate_currents(3.0)
# diff_curr_stack = np.vstack([diff_mode.interpolate_currents(float(c)) for c in diff_ctrl])
# combined_stack = np.where(np.abs(spot_curr)[None, :] > np.abs(diff_curr_stack), spot_curr[None, :], diff_curr_stack)

# il_diff = diff_curr_stack[:, il_idx]
# il_combined = combined_stack[:, il_idx]
# il_override = np.abs(il_combined - il_diff)

# print('\nMode-combination audit (largest-|I| merge used by build_components):')
# for k, name in enumerate(il_names):
#     print(f'  {name}: max |combined - diff_only| = {np.max(il_override[:, k]):.3e} A')

# # 3) Full-column diffraction transfer check (sample-referenced)
# voltage = model_check.voltage
# z_source = model_check.auxiliary.get('z_source', 0.0)
# ray0 = Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=z_source, pathlength=0.0, voltage=voltage)

# ap_info = model_check.auxiliary['apertures']['C_aperture']
# aperture = SigmoidAperture(
#     radius=ap_info['radii'][2],
#     edge_width=ap_info['radii'][2] * 0.05,
#     sharpness=10.0,
#     z=ap_info['z'],
# )
# det_info = model_check.auxiliary['detector']
# detector = Detector(
#     z=det_info['z'],
#     pixel_size=(det_info['pixel_size'], det_info['pixel_size']),
#     shape=(det_info['shape'][0], det_info['shape'][1]),
# )
# samp_info = model_check.auxiliary['sample']
# sample = Plane(z=samp_info['z'])

# z_to_name = {float(l.z_position): l.name for l in model_check.lenses}
# z_to_name[float(aperture.z)] = ap_info.get('name', 'C aperture')
# z_to_name[float(sample.z)] = samp_info.get('name', 'Sample')
# z_to_name[float(detector.z)] = det_info.get('name', 'Detector')
# for d in model_check.deflectors:
#     z_to_name[float(d.z_position)] = d.name

# sample_name = samp_info.get('name', 'Sample')
# detector_name = det_info.get('name', 'Detector')

# def _solve_corot(column, names):
#     _, M_cum, labels, rot = solve_model_with_z(ray0, column, names=names)
#     M_corot = np.stack([
#         _rotation_matrix_5x5(-theta) @ M
#         for theta, M in zip(np.asarray(rot), np.asarray(M_cum))
#     ])
#     return M_corot, labels

# def _diffraction_transfer_metrics(op_builder):
#     ratios = []
#     A_vals = []
#     for cl_target in diff_ctrl:
#         op = op_builder(float(cl_target))
#         optics = list(model_check.build_components(op))
#         col = sorted(optics + [aperture, sample, detector], key=lambda c: float(c.z))
#         names = [z_to_name.get(float(c.z), type(c).__name__) for c in col]

#         M_corot, labels = _solve_corot(col, names)
#         s_idx = max(i for i, lbl in enumerate(labels) if lbl == sample_name)
#         d_idx = max(i for i, lbl in enumerate(labels) if lbl == detector_name)
#         M_sd = M_corot[d_idx] @ np.linalg.inv(M_corot[s_idx])

#         A_vals.append(float(M_sd[0, 0]))
#         ratios.append(abs(float(M_sd[0, 2])) / float(cl_target))

#     ratios = np.asarray(ratios, dtype=float)
#     A_vals = np.asarray(A_vals, dtype=float)
#     sample_cl_scale = float(np.median(ratios))
#     ratio_spread = float(np.max(np.abs(ratios / sample_cl_scale - 1.0)))
#     max_abs_A = float(np.max(np.abs(A_vals)))
#     return sample_cl_scale, ratio_spread, max_abs_A

# scale_diff, spread_diff, maxA_diff = _diffraction_transfer_metrics(lambda cl: {'diff': cl})
# scale_combo, spread_combo, maxA_combo = _diffraction_transfer_metrics(lambda cl: {'spot': 3.0, 'diff': cl})

# print('\nLoaded-model diffraction transfer check (sample-referenced):')
# print('  diff only:')
# print(f'    sample_cl_scale |B|/CL median = {scale_diff:.9f}')
# print(f'    |B|/CL relative spread        = {spread_diff:.3e}')
# print(f'    max |A_sample->det|           = {maxA_diff:.3e}')
# print('  spot + diff (actual operating map):')
# print(f'    sample_cl_scale |B|/CL median = {scale_combo:.9f}')
# print(f'    |B|/CL relative spread        = {spread_combo:.3e}')
# print(f'    max |A_sample->det|           = {maxA_combo:.3e}')

# # 4) Reference-plane diagnostic at one CL: sample->detector vs BFP->detector
# cl_demo = float(diff_ctrl[len(diff_ctrl) // 2])
# op_demo = {'spot': 3.0, 'diff': cl_demo}
# col_demo = sorted(list(model_check.build_components(op_demo)) + [aperture, sample, detector], key=lambda c: float(c.z))
# names_demo = [z_to_name.get(float(c.z), type(c).__name__) for c in col_demo]

# obj_name = 'Obj_post'
# obj_idx = max(i for i, n in enumerate(names_demo) if n == obj_name)
# obj_post = col_demo[obj_idx]
# f_obj = float(obj_post.focal_length(voltage))
# z_bfp = float(obj_post.z) + f_obj

# bfp = Plane(z=z_bfp)
# col_bfp = sorted(col_demo + [bfp], key=lambda c: float(c.z))
# z_to_name_bfp = dict(z_to_name)
# z_to_name_bfp[float(bfp.z)] = 'BFP'
# names_bfp = [z_to_name_bfp.get(float(c.z), type(c).__name__) for c in col_bfp]

# M_corot, labels = _solve_corot(col_bfp, names_bfp)
# s_idx = max(i for i, lbl in enumerate(labels) if lbl == sample_name)
# d_idx = max(i for i, lbl in enumerate(labels) if lbl == detector_name)
# bfp_idx = max(i for i, lbl in enumerate(labels) if lbl == 'BFP')

# M_sd = M_corot[d_idx] @ np.linalg.inv(M_corot[s_idx])
# M_bd = M_corot[d_idx] @ np.linalg.inv(M_corot[bfp_idx])

# A_sd, B_sd = float(M_sd[0, 0]), float(M_sd[0, 2])
# A_bd, B_bd = float(M_bd[0, 0]), float(M_bd[0, 2])

# print('\nReference-plane diagnostic (spot + diff):')
# print(f'  CL demo = {cl_demo:.4f} m, f_obj = {f_obj*1e3:.4f} mm')
# print(f'  sample->detector: A={A_sd:+.6e}, B={B_sd:+.6e} m')
# print(f'  BFP->detector:    A={A_bd:+.6e}, B={B_bd:+.6e} m')
# print('  (Non-zero sample A can coexist with near-zero BFP B, depending on objective state.)')

# if spread_combo < 5e-3 and maxA_combo < 1e-2:
#     print('  status: PASS (combined modes preserve diffraction constraints)')
# else:
#     print('  status: WARN (combined modes deviate from ideal constraints in sample-referenced check)')

## Design-Curve Sanity Plots (from written TOML)

Visual check that the updated `microscope.toml` imaging and diffraction operating tables
match the fitted IL curves used in this notebook.

This is a **table-fidelity** check (fit arrays vs written rows).
It does not by itself guarantee the same transfer-matrix constraints in the full
loaded column model, so the next cell runs an explicit replay audit.

In [ ]:
# Plot design curves from the written TOML against fitted export arrays.

lens_labels = ['IL1', 'IL2', 'IL3']
lens_cols = ['C0', 'C1', 'C2']

# Read back from the in-memory TOML document written in the previous cell.
mag_headers = list(microscope_data['modes']['imaging']['magnification']['headers'])
mag_values = np.asarray(microscope_data['modes']['imaging']['magnification']['values'], dtype=float)
diff_headers = list(microscope_data['modes']['imaging']['diffraction']['headers'])
diff_values = np.asarray(microscope_data['modes']['imaging']['diffraction']['values'], dtype=float)

mag_idx = {k: i for i, k in enumerate(mag_headers)}
diff_idx = {k: i for i, k in enumerate(diff_headers)}

mag_ctrl = mag_values[:, mag_idx['mag']]
mag_toml = np.vstack([mag_values[:, mag_idx[k]] for k in lens_labels])

cl_ctrl_m = diff_values[:, diff_idx['camera_length']]
cl_ctrl_mm = cl_ctrl_m * 1e3
cl_toml = np.vstack([diff_values[:, diff_idx[k]] for k in lens_labels])

fig, axs = plt.subplots(2, 2, figsize=(12, 8))

for i, (lbl, col) in enumerate(zip(lens_labels, lens_cols)):
    axs[0, 0].plot(mags_ref, pi_f_amps[i], 'o--', color=col, alpha=0.75, label=f'{lbl} fit')
    axs[0, 0].plot(mag_ctrl, mag_toml[i], '-', color=col, lw=2.0, label=f'{lbl} TOML')
axs[0, 0].set_xscale('log')
axs[0, 0].set_xlabel('Magnification')
axs[0, 0].set_ylabel('Current [A]')
axs[0, 0].set_title('Imaging design curves')
axs[0, 0].grid(True, alpha=0.3)
axs[0, 0].legend(fontsize=8, ncol=2)

for i, (lbl, col) in enumerate(zip(lens_labels, lens_cols)):
    axs[0, 1].plot(cl_ref, pc_f_amps[i], 'o--', color=col, alpha=0.75, label=f'{lbl} fit')
    axs[0, 1].plot(cl_ctrl_mm, cl_toml[i], '-', color=col, lw=2.0, label=f'{lbl} TOML')
axs[0, 1].set_xscale('log')
axs[0, 1].set_xlabel('Camera length [mm]')
axs[0, 1].set_ylabel('Current [A]')
axs[0, 1].set_title('Diffraction design curves')
axs[0, 1].grid(True, alpha=0.3)
axs[0, 1].legend(fontsize=8, ncol=2)

for i, (lbl, col) in enumerate(zip(lens_labels, lens_cols)):
    axs[1, 0].plot(mags_ref, mag_toml[i] - pi_f_amps[i], 'o-', color=col, label=lbl)
axs[1, 0].axhline(0.0, color='k', lw=1.0, alpha=0.6)
axs[1, 0].set_xscale('log')
axs[1, 0].set_xlabel('Magnification')
axs[1, 0].set_ylabel('TOML - fit [A]')
axs[1, 0].set_title('Imaging write residual')
axs[1, 0].grid(True, alpha=0.3)

for i, (lbl, col) in enumerate(zip(lens_labels, lens_cols)):
    axs[1, 1].plot(cl_ref, cl_toml[i] - pc_f_amps[i], 'o-', color=col, label=lbl)
axs[1, 1].axhline(0.0, color='k', lw=1.0, alpha=0.6)
axs[1, 1].set_xscale('log')
axs[1, 1].set_xlabel('Camera length [mm]')
axs[1, 1].set_ylabel('TOML - fit [A]')
axs[1, 1].set_title('Diffraction write residual')
axs[1, 1].grid(True, alpha=0.3)

fig.suptitle('Microscope design-curve sanity check', fontsize=14)
fig.tight_layout()
plt.show()

print('Max |imaging TOML-fit| [A]:', float(np.max(np.abs(mag_toml - pi_f_amps))))
print('Max |diffraction TOML-fit| [A]:', float(np.max(np.abs(cl_toml - pc_f_amps))))